In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA EXTRACTION (MOCK GENERATOR)
# ==========================================
def fetch_mock_data():
    """
    Simulates extracting daily demand data from Google BigQuery.
    Generates synthetic data representing Milk (1L) demand at an NCR store,
    complete with weekly seasonality and 'payday' demand spikes.
    """
    dates = pd.date_range(start='2025-01-01', end='2026-07-31', freq='D')
    data = []
    
    for d in dates:
        # Base daily demand for Milk
        demand = 120 
        
        # Weekend effect (lower demand for Corporate/CBD stores)
        if d.weekday() >= 5: 
            demand -= 45
            
        # Payday spike (25th to 28th of the month)
        if d.day in [25, 26, 27, 28]:
            demand += 80
            
        # Add random real-world noise/variance
        demand += np.random.normal(0, 12)
        
        data.append({
            'date': d,
            'store_id': 'KK_NCR_001',
            'sku_id': 'SKU_MILK_1L',
            'daily_demand': max(0, int(demand)) # Demand cannot be negative
        })
        
    return pd.DataFrame(data)

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
def engineer_features(df):
    """
    Transforms raw time-series data into a tabular format for XGBoost.
    Adds lags, rolling window statistics, and temporal flags.
    """
    df = df.sort_values(by=['store_id', 'sku_id', 'date']).copy()
    
    # Temporal Features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_payday'] = df['date'].dt.day.isin([25, 26, 27, 28]).astype(int)
    
    # Lags (Looking at past demand)
    grouped = df.groupby(['store_id', 'sku_id'])['daily_demand']
    df['lag_1'] = grouped.shift(1)
    df['lag_7'] = grouped.shift(7)
    df['lag_14'] = grouped.shift(14)
    
    # Rolling Statistics (Capturing momentum and volatility)
    df['rolling_mean_7'] = grouped.rolling(window=7, min_periods=1).mean().reset_index(level=[0,1], drop=True)
    df['rolling_std_7'] = grouped.rolling(window=7, min_periods=1).std().reset_index(level=[0,1], drop=True).fillna(0)
    
    # Drop rows with NaN values created by the shifting process
    df = df.dropna().reset_index(drop=True)
    
    return df

# ==========================================
# 3. MODEL TRAINING & EVALUATION
# ==========================================
def train_and_evaluate(df):
    """
    Splits data chronologically, trains the XGBoost Regressor, 
    and evaluates it against the 15% MAPE threshold target.
    """
    features = ['day_of_week', 'is_weekend', 'is_payday', 
                'lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7']
    target = 'daily_demand'
    
    # Chronological Train-Test Split (Reserve the last 30 days for validation)
    split_date = df['date'].max() - timedelta(days=30)
    
    train_data = df[df['date'] <= split_date]
    val_data = df[df['date'] > split_date]
    
    X_train, y_train = train_data[features], train_data[target]
    X_val, y_val = val_data[features], val_data[target]
    
    # Initialize XGBoost Model with hyperparameters matching the report
    model = xgb.XGBRegressor(
        n_estimators=450,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        objective='reg:squarederror',
        random_state=42
    )
    
    # Train the model
    print("Training XGBoost Model on historical data...")
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    # Generate Predictions
    predictions = model.predict(X_val)
    
    # Calculate Business KPIs
    mape = mean_absolute_percentage_error(y_val, predictions)
    rmse = np.sqrt(mean_squared_error(y_val, predictions))
    
    print(f"\n--- Model Evaluation Results ---")
    print(f"Validation MAPE: {mape:.2%} (Target: < 15%)")
    print(f"Validation RMSE: {rmse:.2f} Units")
    
    # Verify if project objectives are met
    if mape < 0.15:
        print("✅ SUCCESS: Model meets operational accuracy thresholds.")
    else:
        print("❌ WARNING: Model requires further tuning.")
        
    return model, features

# ==========================================
# 4. PIPELINE ORCHESTRATION
# ==========================================
if __name__ == "__main__":
    print("Initializing ETL Pipeline...")
    raw_data = fetch_mock_data()
    print(f"Extracted {len(raw_data)} daily transaction records.")
    
    print("\nExecuting Feature Engineering...")
    processed_data = engineer_features(raw_data)
    
    print("\nStarting MLOps Training Pipeline...")
    trained_model, feature_list = train_and_evaluate(processed_data)
    
    print("\nPipeline execution complete. Model artifacts are ready to be served via REST API.")

Initializing ETL Pipeline...
Extracted 577 daily transaction records.

Executing Feature Engineering...

Starting MLOps Training Pipeline...
Training XGBoost Model on historical data...

--- Model Evaluation Results ---
Validation MAPE: 10.16% (Target: < 15%)
Validation RMSE: 13.01 Units
✅ SUCCESS: Model meets operational accuracy thresholds.

Pipeline execution complete. Model artifacts are ready to be served via REST API.
